# FutureSim: Replaying World Events to Evaluate Adaptive Agents
**ArXivist-generated reproduction notebook**  
Paper: https://arxiv.org/abs/2605.15188  
Generated: 2026-05-15

This notebook walks through the key components of the FutureSim implementation:
the Brier Skill Score metric, the news retrieval system, the simulation engine,
and a mini end-to-end simulation run on synthetic data.

In [ ]:
# Cell 1 — Environment Check
import sys
print(f'Python: {sys.version}')

try:
    import torch
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
except ImportError:
    print('PyTorch not installed (not required for core FutureSim logic)')

try:
    import pandas as pd
    import numpy as np
    print(f'pandas: {pd.__version__}, numpy: {np.__version__}')
except ImportError as e:
    print(f'Missing dependency: {e}')
    print('Run: pip install -r requirements.txt')

In [ ]:
# Cell 2 — Install project (run once)
import subprocess
result = subprocess.run(['pip', 'install', '-e', '..'], capture_output=True, text=True, cwd='..')
if result.returncode == 0:
    print('Installation successful')
else:
    print(result.stderr)

## Paper Overview

**Problem**: AI agents struggle to adapt over long time horizons in dynamic environments.
Standard benchmarks don't test this realistically.

**Core Idea**: FutureSim replays real-world events in chronological order. Agents must
forecast world events *beyond their knowledge cutoff* by reading real news articles that
arrive day by day — just as they would in the real world.

**Key Innovation**: Two minimal actions (`submit_forecast`, `next_day`) expose a
chronological news corpus. Any model + harness combination can be benchmarked.
The environment is fully reproducible (unlike live prediction markets).

**Main Finding**: The best frontier agent (GPT 5.5) achieves only **25% accuracy**
forecasting 330 real-world events over Jan–Mar 2026. Many open-weight models do
worse than abstaining (negative Brier Skill Score).

## Component 1: Brier Skill Score

FutureSim uses an extended Brier Skill Score for free-form multi-outcome predictions
(Section 3, Appendix C.2).

**Formula** (Equation from Section 3):
$$BSS(q) = 1 - \sum_{o \in \Omega_q \cup \{y_q\}} \left( p_q(o) - \mathbf{1}[o = y_q] \right)^2$$

This is a **proper scoring rule**: agents maximize expected score by reporting their true beliefs.
- $BSS = 1$: fully confident correct answer  
- $BSS = 0$: abstaining (no prediction submitted)  
- $BSS = -1$: all probability mass on wrong answers

In [ ]:
# Demonstrate BSS with the worked example from Appendix C.1
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve() / 'src'))

from futuresim.scoring.brier import (
    compute_brier_skill_score, compute_accuracy, compute_total_variation_distance
)

ground_truth = 'Seattle Seahawks'

# Forecast A: confident and correct (Appendix C.1)
forecast_a = {'Seattle Seahawks': 0.55, 'Chiefs': 0.25, '49ers': 0.10}
bss_a = compute_brier_skill_score(forecast_a, ground_truth)
print(f'Forecast A (confident correct): BSS = {bss_a:.4f}')   # Expected: 0.725

# Forecast B: dominant wrong answer despite listing correct one (Appendix C.1)
forecast_b = {'Chiefs': 0.55, 'Bills': 0.20, 'Seattle Seahawks': 0.15, '49ers': 0.10}
bss_b = compute_brier_skill_score(forecast_b, ground_truth)
print(f'Forecast B (confident wrong):   BSS = {bss_b:.4f}')   # Expected: -0.075

# Abstaining
bss_abstain = compute_brier_skill_score({}, ground_truth)
print(f'Abstain (no prediction):        BSS = {bss_abstain:.4f}')  # Expected: 0.0

print()
print('Accuracy (top-1):')
print(f'  Forecast A: {compute_accuracy(forecast_a, ground_truth)}')  # 1.0
print(f'  Forecast B: {compute_accuracy(forecast_b, ground_truth)}')  # 0.0

## Component 2: Hybrid News Retrieval

FutureSim uses a **hybrid semantic + keyword search** over a date-gated CCNews corpus
(Section 4.1, Appendix B.5):

- **7.36M articles** from 141 sources (Jan 2023 – Mar 2026)
- **Qwen3-8B embeddings** for semantic search
- **LanceDB** for hybrid retrieval (semantic + BM25 keyword)
- Returns **5 chunks of 512 tokens** per query
- **Date-gated**: `to_date` capped at simulation date (prevents future leakage)

```python
search_news(query, from_date, to_date)  # agent-facing API
```

In [ ]:
# Demo: Retriever interface (requires LanceDB index — shows mock if unavailable)
try:
    from futuresim.corpus.retrieval import NewsRetriever
    retriever = NewsRetriever(
        index_path='../data/lancedb_index/',
        embedding_model_name='Qwen/Qwen3-Embedding-8B',
        chunks_per_query=5,
        chunk_size=512,
    )
    print(f'Retriever initialized: {retriever}')
    print('Note: Call retriever.connect() and retriever.set_date_cap(date) before searching.')
    print('      Requires a pre-built LanceDB index (see build_index.py)')
except ImportError as e:
    print(f'Optional dependency not installed: {e}')
    print('Retriever interface demonstrated without index.')

## Component 3: Multi-Agent Aggregate & TV Distance

For multi-agent experiments (Section 5.5), FutureSim computes a **crowd aggregate**
as the coordinate-wise mean of all agents' predictions (Appendix C.4):

$$\bar{p}_q(o) = \frac{1}{n_q} \sum_{a=1}^{n_q} p_q^{(a)}(o)$$

And measures divergence via **Total Variation Distance**:

$$d_{TV}(p_q, p'_q) = \frac{1}{2} \sum_{o \in \Omega_q \cup \Omega'_q} |p_q(o) - p'_q(o)|$$

In [ ]:
from futuresim.scoring.brier import compute_aggregate_prediction, compute_total_variation_distance

# Simulate 3 agents predicting the Nepal PM question
agent_predictions = [
    {'Balendra Shah': 0.45, 'Gagan Thapa': 0.35, 'KP Sharma Oli': 0.15},
    {'Gagan Thapa': 0.50, 'Balendra Shah': 0.30, 'Pushpa Kamal Dahal': 0.10},
    {'Balendra Shah': 0.60, 'Gagan Thapa': 0.25},
]

aggregate = compute_aggregate_prediction(agent_predictions)
print('Crowd aggregate prediction:')
for outcome, prob in sorted(aggregate.items(), key=lambda x: -x[1]):
    print(f'  {outcome}: {prob:.3f}')

# TV distance between agent 0 and aggregate
tv = compute_total_variation_distance(agent_predictions[0], aggregate)
print(f'\nTV distance (Agent 0 vs aggregate): {tv:.4f}')

## Component 4: Mini End-to-End Simulation

FutureSim's simulation loop (Section 3.1):
1. Each day: agent searches news, submits/updates forecasts via `submit_forecast()`
2. Agent calls `next_day()` to advance simulation
3. Engine resolves questions due that day, computes BSS, delivers feedback

Below we run a minimal synthetic simulation to verify the engine works correctly.

In [ ]:
# Mini simulation with synthetic data (no downloads required)
import json
import tempfile
import pathlib
from datetime import date, timedelta

# Build synthetic questions CSV
import pandas as pd

today = date.today()
questions = pd.DataFrame([
    {
        'qid': 0,
        'title': 'Who won the synthetic championship?',
        'background': 'A fictional sporting event.',
        'resolution_criteria': 'Name of the winning team.',
        'answer_type': 'String (Name)',
        'resolution_date': str(today + timedelta(days=2)),
        'is_resolved': False,
        'ground_truth': None,
        'my_prediction': None,
        'my_prediction_date': None,
        'num_predictions': 0,
    },
    {
        'qid': 1,
        'title': 'What will be the outcome of the synthetic summit?',
        'background': 'A fictional diplomatic event.',
        'resolution_criteria': 'Outcome string.',
        'answer_type': 'String',
        'resolution_date': str(today + timedelta(days=3)),
        'is_resolved': False,
        'ground_truth': None,
        'my_prediction': None,
        'my_prediction_date': None,
        'num_predictions': 0,
    },
])

print(f'Synthetic questions:')
print(questions[['qid', 'title', 'resolution_date']].to_string(index=False))

In [ ]:
# Validate BSS against paper's worked example (Appendix C.1)
# Forecast A: BSS should be 0.725
fa = {'Seattle Seahawks': 0.55, 'Chiefs': 0.25, '49ers': 0.10}
bss_a = compute_brier_skill_score(fa, 'Seattle Seahawks')
expected_a = 0.725
assert abs(bss_a - expected_a) < 1e-4, f'BSS Forecast A: {bss_a:.4f} != {expected_a}'

# Forecast B: BSS should be -0.075
fb = {'Chiefs': 0.55, 'Bills': 0.20, 'Seattle Seahawks': 0.15, '49ers': 0.10}
bss_b = compute_brier_skill_score(fb, 'Seattle Seahawks')
expected_b = -0.075
assert abs(bss_b - expected_b) < 1e-4, f'BSS Forecast B: {bss_b:.4f} != {expected_b}'

# Abstain: BSS should be 0.0
bss_abstain = compute_brier_skill_score({}, 'Seattle Seahawks')
assert abs(bss_abstain) < 1e-9, f'BSS Abstain: {bss_abstain}'

print('All BSS computations match paper Appendix C.1 ✓')
print(f'  Forecast A: {bss_a:.4f} (expected {expected_a})')
print(f'  Forecast B: {bss_b:.4f} (expected {expected_b})')
print(f'  Abstain:    {bss_abstain:.4f} (expected 0.0)')

## Paper Results (Table from Section 4.2, Figure 1)

FutureSim evaluated 5 frontier agents on 330 questions over Jan–Mar 2026.

In [ ]:
# Reported results from paper (Section 4.2, Figure 1)
paper_results = {
    'GPT 5.5':         {'harness': 'Codex',       'accuracy': 0.25, 'brier_skill_score': 0.05},
    'Claude Opus 4.6': {'harness': 'Claude Code', 'accuracy': 0.20, 'brier_skill_score': 0.02},
    'DeepSeek V4 Pro': {'harness': 'Claude Code', 'accuracy': 0.13, 'brier_skill_score': -0.02},
    'GLM 5.1':         {'harness': 'Claude Code', 'accuracy': 0.10, 'brier_skill_score': -0.01},
    'Qwen 3.6 Plus':   {'harness': 'OpenCode',    'accuracy': 0.05, 'brier_skill_score': -0.07},
}

print('Paper Results (Figure 1, recommended harness, 3-seed mean):')
print(f'{"Model":<20} {"Harness":<15} {"Accuracy":>10} {"BSS":>10}')
print('-' * 57)
for model, r in paper_results.items():
    print(f'{model:<20} {r["harness"]:<15} {r["accuracy"]:>9.0%} {r["brier_skill_score"]:>10.2f}')

print()
print('Key findings:')
print('  - Best accuracy: 25% (GPT 5.5) — substantial room for improvement')
print('  - 3 of 5 models have negative BSS — worse than abstaining')
print('  - All models improve accuracy over time as news arrives')
print('  - Custom harness improves open-weight model performance significantly')

## What to Do Next

1. **Build the CCNews index**: `python build_index.py --corpus-path data/ccnews/ --index-path data/lancedb_index/`
2. **Run the simulation**: `python run_simulation.py --config configs/config.yaml --agent native --model gpt-4o --seed 0`
3. **Score results**: `python score_results.py --predictions results/predictions.json --ground-truth data/ground_truth.json`
4. **Use the Results Comparator (Stage 6)**: Feed your results back to ArXivist

**Implementation notes from the SIR (low-confidence sections):**
- ⚠️ Hybrid retrieval fusion method not specified (confidence 0.55) — using LanceDB default
- ⚠️ Qwen3-8B embedding dimension not stated (confidence 0.62) — assumed 4096
- ⚠️ bwrap sandbox full configuration not provided (confidence 0.65) — standard namespace isolation assumed

**Reproducibility note**: The paper evaluated GPT 5.5 and Opus 4.6 via provider coding plans
(~$220/month). Direct API equivalent would cost substantially more.
See Appendix B.4 for full cost details.